# 🛡️ Time-Series to Graph-Based Intrusion Detection in IoT/IIoT Networks
## UNSW-NB15 Dataset · Kaggle Version (GPU-Enabled)

---

| | |
|---|---|
| **Course** | AIML505 – Statistics for Data Science |
| **Theme** | Time-Series & Graph Neural Networks |
| **Dataset** | UNSW-NB15 (ACCS, University of New South Wales) |
| **Runtime** | Kaggle — GPU P100 or T4 recommended |

---

## ⚡ Kaggle Setup Instructions

Before running this notebook on Kaggle, complete these three steps:

### Step 1 — Upload the Dataset to Kaggle
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Upload both files:
   - `UNSW_NB15_training-set.csv`
   - `UNSW_NB15_testing-set.csv`
3. Name the dataset: **`unsw-nb15`**
4. After creation, note your Kaggle username.

### Step 2 — Add Dataset to This Notebook
In the Kaggle notebook editor:
- Click **⊕ Add Data** (right sidebar)
- Search for your uploaded dataset (`unsw-nb15`)
- Click **Add** — files will appear at `/kaggle/input/unsw-nb15/`

### Step 3 — Enable GPU
- Click **Settings** (⚙) → **Accelerator** → Select **GPU T4 x2** or **GPU P100**
- Click **Save**

### Step 4 — Run All Cells
- Click **Run All** or use Shift+Enter on each cell.

---

## Abstract

This research notebook implements a dual-track intrusion detection system (IDS) for IoT/IIoT environments:
- **Track A:** Classical ML + Sequential Deep Learning (LSTM, GRU, BiLSTM, CNN-LSTM)
- **Track B:** Graph Neural Networks (GCN, GraphSAGE, GAT, GIN) via PyTorch Geometric

Both tracks are compared on identical preprocessing pipelines with full metric suites.


## 2. Imports & Package Setup

On Kaggle, PyTorch Geometric (PyG) and Optuna can be installed directly using pip. We detect the Kaggle environment automatically and install them if not present.

In [ ]:
import os, sys, time, random, warnings, subprocess, json
from copy import deepcopy
from collections import Counter
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import kneighbors_graph, KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, IsolationForest)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve, auc,
                             confusion_matrix, classification_report)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.figsize": (12, 5), "axes.titlesize": 14, "figure.dpi": 100})

# ── Environment Detection ─────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/input')
print(f"Running on Kaggle: {ON_KAGGLE}")

# ── Auto-install helper ────────────────────────────────────────────────────────
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print(f'  ✔ Installed: {pkg}')

# ── Install Optuna ─────────────────────────────────────────────────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('✔ Optuna already available')
except ImportError:
    pip_install('optuna')
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Install UMAP ──────────────────────────────────────────────────────────────
UMAP_AVAILABLE = False
try:
    import umap
    UMAP_AVAILABLE = True
    print('✔ UMAP already available')
except ImportError:
    try:
        pip_install('umap-learn')
        import umap
        UMAP_AVAILABLE = True
        print('✔ UMAP installed')
    except Exception:
        print('⚠ UMAP unavailable — skipping UMAP plot')

# ── Install tabulate ──────────────────────────────────────────────────────────
try:
    from tabulate import tabulate
except ImportError:
    pip_install('tabulate')
    from tabulate import tabulate

# ── Install PyTorch Geometric (Kaggle/GPU-friendly) ──────────────────────────
GNN_BACKEND = 'custom'
try:
    import torch_geometric
    from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINConv
    from torch_geometric.data import Data
    GNN_BACKEND = 'pyg'
    print('✔ PyTorch Geometric already available')
except ImportError:
    print('Installing PyTorch Geometric for Kaggle ...')
    try:
        import torch
        torch_ver = torch.__version__.split('+')[0]
        cuda_tag  = 'cu118' if torch.cuda.is_available() else 'cpu'
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            'torch_geometric',
            'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
            '-f', f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html',
            '-q'
        ])
        import torch_geometric
        from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINConv
        from torch_geometric.data import Data
        GNN_BACKEND = 'pyg'
        print('✔ PyTorch Geometric installed successfully')
    except Exception as e:
        print(f'⚠ PyG install failed ({e}) → using custom PyTorch GNN layers (fully functional fallback)')

# ── Device detection ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'✔ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU detected — using CPU (enable GPU in Kaggle Settings → Accelerator)')

print(f'\n✅ All imports complete | Device: {DEVICE} | GNN Backend: {GNN_BACKEND}')


## 3. Global Configuration

Kaggle provides ~16 GB RAM and GPU. We increase sample sizes, epoch counts, and enable GPU acceleration relative to the CPU-local version.

In [ ]:
class Config:
    """Centralised configuration for the AIML505 Kaggle experiment.
    Kaggle-optimised: larger samples, more epochs, GPU-aware.
    """
    # ── Dataset Paths ─────────────────────────────────────────────────────────
    # Kaggle path (after adding the unsw-nb15 dataset)
    KAGGLE_BASE  = '/kaggle/input/unsw-nb15'
    LOCAL_BASE   = 'Training and Testing Sets'

    @staticmethod
    def get_paths():
        if os.path.exists('/kaggle/input/unsw-nb15'):
            base = '/kaggle/input/unsw-nb15'
        elif os.path.exists('Training and Testing Sets'):
            base = 'Training and Testing Sets'
        else:
            raise FileNotFoundError(
                'Dataset not found. On Kaggle: Add Data → unsw-nb15. '
                'Locally: place CSVs in Training and Testing Sets/'
            )
        return (f'{base}/UNSW_NB15_training-set.csv',
                f'{base}/UNSW_NB15_testing-set.csv')

    # ── Reproducibility ───────────────────────────────────────────────────────
    SEED = 42

    # ── Hardware ──────────────────────────────────────────────────────────────
    DEVICE = DEVICE

    # ── Sampling (larger on Kaggle GPU) ──────────────────────────────────────
    TRAIN_SAMPLE = 50_000 if DEVICE == 'cuda' else 20_000
    TEST_SAMPLE  = 15_000 if DEVICE == 'cuda' else  5_000
    DL_TRAIN     = 20_000 if DEVICE == 'cuda' else  8_000
    DL_TEST      =  5_000 if DEVICE == 'cuda' else  2_000
    GNN_NODES    = 10_000 if DEVICE == 'cuda' else  3_000

    # ── Sliding window ────────────────────────────────────────────────────────
    WINDOW_SIZE  = 10 if DEVICE == 'cuda' else 5

    # ── Deep Learning ─────────────────────────────────────────────────────────
    DL_EPOCHS    = 30 if DEVICE == 'cuda' else 10
    DL_BATCH     = 256 if DEVICE == 'cuda' else 64
    DL_LR        = 1e-3
    DL_HIDDEN    = 64 if DEVICE == 'cuda' else 32
    DL_PATIENCE  = 5

    # ── GNN ───────────────────────────────────────────────────────────────────
    GNN_EPOCHS      = 50 if DEVICE == 'cuda' else 20
    GNN_LR          = 1e-3
    GNN_HIDDEN      = 64 if DEVICE == 'cuda' else 32
    GNN_K_NEIGHBORS = 8 if DEVICE == 'cuda' else 5

    # ── Correlation threshold ──────────────────────────────────────────────────
    CORR_THRESHOLD = 0.95

CFG = Config()
TRAIN_PATH, TEST_PATH = CFG.get_paths()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)

print(f'✅ Config loaded | Device: {CFG.DEVICE} | Train Sample: {CFG.TRAIN_SAMPLE:,}')
print(f'   Train path: {TRAIN_PATH}')
print(f'   Test  path: {TEST_PATH}')


## 4. Dataset Loading

In [ ]:
df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

print(f'✅ Training set: {df_train_raw.shape[0]:,} rows × {df_train_raw.shape[1]} cols')
print(f'✅ Testing  set: {df_test_raw.shape[0]:,} rows × {df_test_raw.shape[1]} cols')
df_train_raw.head(3)


## 5. Dataset Inspection

Dynamic schema auto-detection — no hardcoded column names.

In [ ]:
def inspect_dataset(df_tr, df_te):
    schema = {}
    schema['binary_target']     = 'label'      if 'label'      in df_tr.columns else df_tr.columns[-1]
    schema['multiclass_target'] = 'attack_cat' if 'attack_cat' in df_tr.columns else None
    schema['id_col']            = 'id'         if 'id'         in df_tr.columns else None
    schema['missing_train'] = int(df_tr.isnull().sum().sum())
    schema['missing_test']  = int(df_te.isnull().sum().sum())
    schema['dup_train']     = int(df_tr.duplicated().sum())
    schema['dup_test']      = int(df_te.duplicated().sum())
    exclude = [c for c in [schema['binary_target'], schema['multiclass_target'],
                            schema['id_col']] if c]
    schema['numeric_cols'] = [c for c in df_tr.select_dtypes(include=[np.number]).columns if c not in exclude]
    schema['cat_cols']     = [c for c in df_tr.select_dtypes(exclude=[np.number]).columns if c not in exclude]
    if schema['multiclass_target']:
        schema['attack_cats'] = df_tr[schema['multiclass_target']].unique().tolist()
    return schema

SCHEMA = inspect_dataset(df_train_raw, df_test_raw)
TARGET   = SCHEMA['binary_target']
MCTARGET = SCHEMA['multiclass_target']

print('─' * 55)
print(f"  Binary target     : {TARGET}")
print(f"  Multiclass target : {MCTARGET}")
print(f"  Numerical features: {len(SCHEMA['numeric_cols'])}")
print(f"  Categorical feat. : {SCHEMA['cat_cols']}")
print(f"  Missing (train)   : {SCHEMA['missing_train']}")
print(f"  Duplicates (train): {SCHEMA['dup_train']}")
print(f"  Attack categories : {SCHEMA.get('attack_cats','N/A')}")
print('─' * 55)


## 6. Exploratory Data Analysis (EDA)

In [ ]:
df_eda = df_train_raw.sample(n=min(CFG.TRAIN_SAMPLE, len(df_train_raw)), random_state=CFG.SEED)

# ── Target & Attack Distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vc = df_eda[TARGET].value_counts()
axes[0].bar(['Normal (0)', 'Attack (1)'], vc.values, color=['#2196F3','#F44336'], edgecolor='white')
axes[0].set_title('Binary Label Distribution')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/vc.sum()*100:.1f}%)', ha='center', fontweight='bold')
if MCTARGET:
    cat_order = df_eda[MCTARGET].value_counts()
    axes[1].barh(cat_order.index, cat_order.values, color=sns.color_palette('husl', len(cat_order)))
    axes[1].set_title('Attack Category Distribution'); axes[1].invert_yaxis()
proto_vc = df_eda['proto'].value_counts().head(10)
axes[2].bar(proto_vc.index, proto_vc.values, color=sns.color_palette('Set2', len(proto_vc)))
axes[2].set_title('Top-10 Protocols'); axes[2].tick_params(axis='x', rotation=45)
plt.suptitle('UNSW-NB15 — Overview', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Feature Histograms (KDE) ──────────────────────────────────────────────────
key_feats = [f for f in ['dur','sbytes','dbytes','rate','sttl','dttl'] if f in df_eda.columns]
fig, axes = plt.subplots(2, 3, figsize=(18, 9)); axes = axes.flatten()
for i, feat in enumerate(key_feats):
    d0 = np.log1p(df_eda[df_eda[TARGET]==0][feat].dropna())
    d1 = np.log1p(df_eda[df_eda[TARGET]==1][feat].dropna())
    axes[i].hist(d0, bins=40, alpha=0.5, label='Normal', color='#2196F3', density=True)
    axes[i].hist(d1, bins=40, alpha=0.5, label='Attack',  color='#F44336', density=True)
    d0.plot.kde(ax=axes[i], color='#0D47A1', lw=2)
    d1.plot.kde(ax=axes[i], color='#B71C1C', lw=2)
    axes[i].set_title(f'log(1+{feat})'); axes[i].legend(fontsize=9)
plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Violin Plots ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, feat in enumerate(['dur','rate','sbytes']):
    if feat in df_eda.columns:
        tmp = df_eda[[TARGET, feat]].copy(); tmp[feat] = np.log1p(tmp[feat])
        sns.violinplot(data=tmp, x=TARGET, y=feat, ax=axes[i],
                       palette=['#2196F3','#F44336'], inner='box')
        axes[i].set_title(f'log(1+{feat}) by Class')
        axes[i].set_xticklabels(['Normal','Attack'])
plt.suptitle('Violin Plots', fontsize=14, fontweight='bold'); plt.tight_layout(); plt.show()

# ── PCA + t-SNE + UMAP ───────────────────────────────────────────────────────
DIM_N = 1500
df_dim = df_eda.sample(n=min(DIM_N, len(df_eda)), random_state=CFG.SEED)
NCOLS  = SCHEMA['numeric_cols']
X_dim  = df_dim[NCOLS].fillna(0).values
y_dim  = df_dim[TARGET].values
X_sc   = StandardScaler().fit_transform(X_dim)

pca   = PCA(n_components=2, random_state=CFG.SEED)
X_pca = pca.fit_transform(X_sc)
print('Running t-SNE ...'); tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=CFG.SEED, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_sc)

n_plots = 3 if UMAP_AVAILABLE else 2
fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 6))
for ax, X2, title in zip(axes[:2], [X_pca, X_tsne], ['PCA (2D)', 't-SNE (2D)']):
    ax.scatter(X2[y_dim==0,0], X2[y_dim==0,1], s=12, alpha=0.5, c='#2196F3', label='Normal')
    ax.scatter(X2[y_dim==1,0], X2[y_dim==1,1], s=12, alpha=0.5, c='#F44336', label='Attack')
    ax.set_title(title, fontweight='bold'); ax.legend(markerscale=2)
if UMAP_AVAILABLE:
    print('Running UMAP ...'); reducer = umap.UMAP(n_components=2, random_state=CFG.SEED)
    X_umap = reducer.fit_transform(X_sc)
    axes[2].scatter(X_umap[y_dim==0,0], X_umap[y_dim==0,1], s=12, alpha=0.5, c='#2196F3', label='Normal')
    axes[2].scatter(X_umap[y_dim==1,0], X_umap[y_dim==1,1], s=12, alpha=0.5, c='#F44336', label='Attack')
    axes[2].set_title('UMAP (2D)', fontweight='bold'); axes[2].legend(markerscale=2)
plt.suptitle('Dimensionality Reduction', fontsize=15, fontweight='bold'); plt.tight_layout(); plt.show()
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')


## 7. Statistical Analysis

### 7.1 Descriptive Statistics, Normality & Hypothesis Testing

In [ ]:
# Descriptive stats
desc = df_eda[SCHEMA['numeric_cols'][:8]].describe().T
desc['skewness'] = df_eda[SCHEMA['numeric_cols'][:8]].skew()
desc['kurtosis'] = df_eda[SCHEMA['numeric_cols'][:8]].kurtosis()
print('Descriptive Statistics:')
print(desc[['mean','std','min','50%','max','skewness','kurtosis']].to_string())

# KS normality test
print('\nKolmogorov-Smirnov Normality Tests:')
print(f'{"Feature":<25} {"KS":>8} {"p-value":>12} {"Normal?":>8}')
print('-' * 57)
for feat in SCHEMA['numeric_cols'][:12]:
    s = df_eda[feat].dropna().values
    ks, pv = stats.kstest((s - s.mean())/(s.std()+1e-9), 'norm')
    print(f'{feat:<25} {ks:>8.4f} {pv:>12.4e} {"No" if pv<0.05 else "Yes":>8}')

# Mann-Whitney U
print('\nMann-Whitney U Test (Normal vs Attack):')
for feat in SCHEMA['numeric_cols'][:10]:
    g0 = df_eda[df_eda[TARGET]==0][feat].dropna()
    g1 = df_eda[df_eda[TARGET]==1][feat].dropna()
    if len(g0)>10 and len(g1)>10:
        u, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        r    = 1 - 2*u/(len(g0)*len(g1))
        print(f'  {feat:<25} effect_size={abs(r):.4f}  p={p:.2e}  {"✓ Significant" if p<0.05 else "✗"}')


### 7.2 Correlation Heatmap & Stationarity Tests

In [ ]:
# Correlation heatmap
corr = df_eda[SCHEMA['numeric_cols']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.4, ax=ax)
ax.set_title('Pearson Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ADF & KPSS
series = df_train_raw['rate'].fillna(0).values[:1000]
adf_s, adf_p, *_ = adfuller(series, autolag='AIC')
kpss_s, kpss_p, *_ = kpss(series, regression='c', nlags='auto')
print(f'ADF  → stat={adf_s:.4f}  p={adf_p:.4f}  → {"Stationary" if adf_p<0.05 else "Non-stationary"}')
print(f'KPSS → stat={kpss_s:.4f}  p={kpss_p:.4f}  → {"Stationary" if kpss_p>0.05 else "Non-stationary"}')

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(series,  lags=40, ax=axes[0], color='steelblue')
plot_pacf(series, lags=40, ax=axes[1], color='tomato')
plt.suptitle("ACF / PACF — Traffic Rate", fontweight='bold'); plt.tight_layout(); plt.show()


## 8. Data Cleaning

In [ ]:
def clean_dataframe(df, schema):
    df = df.copy()
    if schema['id_col'] and schema['id_col'] in df.columns:
        df.drop(columns=[schema['id_col']], inplace=True)
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
    for col in df.select_dtypes(exclude=[np.number]).columns:
        if df[col].isnull().any():
            df[col].fillna(df[col].mode()[0], inplace=True)
    before = len(df); df.drop_duplicates(inplace=True)
    if before != len(df): print(f'  Dropped {before-len(df)} duplicates.')
    return df.reset_index(drop=True)

df_train_clean = clean_dataframe(df_train_raw, SCHEMA)
df_test_clean  = clean_dataframe(df_test_raw,  SCHEMA)
print(f'✅ Train: {df_train_clean.shape}  |  Test: {df_test_clean.shape}')


## 9. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    if 'sbytes' in df.columns and 'dbytes' in df.columns:
        df['bytes_ratio'] = df['sbytes'] / (df['dbytes'] + 1.0)
        df['sbytes_rate'] = df['sbytes'] * df.get('rate', pd.Series(np.zeros(len(df))))
        df['dbytes_rate'] = df['dbytes'] * df.get('rate', pd.Series(np.zeros(len(df))))
    if 'spkts' in df.columns and 'dpkts' in df.columns:
        df['pkts_ratio']  = df['spkts'] / (df['dpkts'] + 1.0)
        df['total_pkts']  = df['spkts'] + df['dpkts']
        df['total_bytes'] = df.get('sbytes', 0) + df.get('dbytes', 0)
    if 'sjit' in df.columns and 'djit' in df.columns:
        df['jit_ratio'] = df['sjit'] / (df['djit'] + 1e-6)
    if 'dur' in df.columns:
        df['rolling_mean_dur'] = df['dur'].rolling(3, min_periods=1).mean()
        df['rolling_std_dur']  = df['dur'].rolling(3, min_periods=1).std().fillna(0)
        df['ema_dur']          = df['dur'].ewm(span=5, adjust=False).mean()
    if 'rate' in df.columns:
        df['rolling_mean_rate'] = df['rate'].rolling(5, min_periods=1).mean()
        df['rolling_std_rate']  = df['rate'].rolling(5, min_periods=1).std().fillna(0)
    for col in ['proto','state','service']:
        if col in df.columns:
            freq = df[col].value_counts() / len(df)
            df[f'{col}_freq'] = df[col].map(freq).fillna(0)
    return df

df_train_eng = engineer_features(df_train_clean)
df_test_eng  = engineer_features(df_test_clean)
new_feats = [c for c in df_train_eng.columns if c not in df_train_clean.columns]
print(f'✅ {len(new_feats)} engineered features: {new_feats}')


## 10. Data Encoding & 11. Scaling

In [ ]:
def encode_dataframes(df_tr, df_te, schema):
    df_tr, df_te = df_tr.copy(), df_te.copy()
    cat_features = df_tr.select_dtypes(exclude=[np.number]).columns.tolist()
    if schema['multiclass_target'] in cat_features:
        cat_features.remove(schema['multiclass_target'])
    for col in cat_features:
        le = LabelEncoder()
        df_tr[col] = le.fit_transform(df_tr[col].astype(str))
        le_map = dict(zip(le.classes_, le.transform(le.classes_)))
        df_te[col] = df_te[col].apply(lambda x: le_map.get(str(x), -1))
    return df_tr, df_te

df_train_enc, df_test_enc = encode_dataframes(df_train_eng, df_test_eng, SCHEMA)

def get_feat_cols(df, schema):
    excl = [schema['binary_target'], schema['multiclass_target'], schema['id_col']]
    return [c for c in df.columns if c not in excl and c is not None]

FEAT_COLS = get_feat_cols(df_train_enc, SCHEMA)
scaler = MinMaxScaler()
df_train_enc[FEAT_COLS] = scaler.fit_transform(df_train_enc[FEAT_COLS])
df_test_enc[FEAT_COLS]  = scaler.transform(df_test_enc[FEAT_COLS])
print(f'✅ Encoded & scaled {len(FEAT_COLS)} feature columns.')


## 12. Class Imbalance Analysis

In [ ]:
y_full  = df_train_enc[TARGET].values
counts  = Counter(y_full)
n_total = len(y_full)
class_weight_dict = {cls: n_total/(2.0*cnt) for cls, cnt in counts.items()}
print(f'Normal (0): {counts[0]:,} ({counts[0]/n_total*100:.1f}%)')
print(f'Attack (1): {counts[1]:,} ({counts[1]/n_total*100:.1f}%)')
print(f'Class weights: {class_weight_dict}')


## 13. Correlation Analysis & Feature Selection

In [ ]:
corr_mat = df_train_enc[FEAT_COLS].corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
HIGH_CORR = [col for col in upper.columns if any(upper[col] > CFG.CORR_THRESHOLD)]
FEAT_COLS_FINAL = [c for c in FEAT_COLS if c not in HIGH_CORR]
print(f'Dropped {len(HIGH_CORR)} high-correlation features.')
print(f'Final feature set: {len(FEAT_COLS_FINAL)} features.')

# RF Feature Importance
rf_imp = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=CFG.SEED, n_jobs=-1)
s_idx  = np.random.choice(len(df_train_enc), size=min(8000, len(df_train_enc)), replace=False)
rf_imp.fit(df_train_enc[FEAT_COLS_FINAL].values[s_idx], df_train_enc[TARGET].values[s_idx])
imp_df = pd.Series(rf_imp.feature_importances_, index=FEAT_COLS_FINAL).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
imp_df.head(20).plot.bar(ax=ax, color=sns.color_palette('viridis', 20))
ax.set_title('Top-20 Feature Importances (Random Forest)', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout(); plt.show()


## 14. Stationarity Analysis & 15. Time-Series Preparation

In [ ]:
# Stationarity on rolling features
for col in ['rolling_mean_dur','rolling_mean_rate','ema_dur']:
    if col in df_train_enc.columns:
        s = df_train_enc[col].dropna().values[:800]
        adf_s, adf_p, *_ = adfuller(s)
        print(f"ADF on '{col}': p={adf_p:.4f} → {'Stationary' if adf_p<0.05 else 'Non-stationary'}")

# Sliding window sequences
def create_sequences(X, y, window=5):
    Xs, ys = [], []
    for i in range(len(X) - window + 1):
        Xs.append(X[i:i+window]); ys.append(y[i+window-1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

dl_tr = df_train_enc.sample(n=min(CFG.DL_TRAIN, len(df_train_enc)), random_state=CFG.SEED).sort_index()
dl_te = df_test_enc.sample(n=min(CFG.DL_TEST,  len(df_test_enc)),  random_state=CFG.SEED).sort_index()

X_seq_tr, y_seq_tr = create_sequences(dl_tr[FEAT_COLS_FINAL].values.astype(np.float32), dl_tr[TARGET].values.astype(np.float32), CFG.WINDOW_SIZE)
X_seq_te, y_seq_te = create_sequences(dl_te[FEAT_COLS_FINAL].values.astype(np.float32), dl_te[TARGET].values.astype(np.float32), CFG.WINDOW_SIZE)
N_FEATURES = X_seq_tr.shape[2]
print(f'✅ Sequences: Train {X_seq_tr.shape}, Test {X_seq_te.shape}')


## 16. Graph Construction & 17. Graph Visualisation

In [ ]:
import networkx as nx
from sklearn.neighbors import kneighbors_graph

gnn_df  = df_train_enc.sample(n=min(CFG.GNN_NODES, len(df_train_enc)), random_state=CFG.SEED).sort_index()
X_graph = gnn_df[FEAT_COLS_FINAL].values.astype(np.float32)
y_graph = gnn_df[TARGET].values.astype(np.float32)

print(f'Building {CFG.GNN_K_NEIGHBORS}-NN graph on {len(y_graph)} nodes ...')
A_sparse = kneighbors_graph(X_graph, n_neighbors=CFG.GNN_K_NEIGHBORS,
                             mode='connectivity', include_self=False, n_jobs=-1)
A_sym = A_sparse + A_sparse.T
A_sym.data = np.ones_like(A_sym.data)

# For PyG we use edge_index; for dense we use A_dense
edge_index = torch.tensor(np.array(A_sym.nonzero()), dtype=torch.long)
A_dense    = torch.FloatTensor(A_sym.toarray())
print(f'✅ Graph: {len(y_graph)} nodes, {edge_index.shape[1]} edges')

# Visualise sub-graph
VIS_N = 200
A_sub = A_dense[:VIS_N, :VIS_N].numpy()
y_sub = y_graph[:VIS_N]
G = nx.from_numpy_array(A_sub)
pos = nx.spring_layout(G, seed=CFG.SEED)
colors = ['#2196F3' if l==0 else '#F44336' for l in y_sub]
fig, ax = plt.subplots(figsize=(12, 9))
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=60, alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.12, edge_color='#90A4AE', ax=ax)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0], marker='o', color='w', markerfacecolor='#2196F3', markersize=10, label='Normal'),
                   Line2D([0],[0], marker='o', color='w', markerfacecolor='#F44336', markersize=10, label='Attack')],
          fontsize=11, loc='upper left')
ax.set_title(f'k-NN Similarity Graph — {VIS_N}-Node Sub-graph', fontsize=14, fontweight='bold')
ax.axis('off'); plt.tight_layout(); plt.show()


## 18. Track A — Classical Machine Learning Models

In [ ]:
ml_tr = df_train_enc.sample(n=min(CFG.TRAIN_SAMPLE, len(df_train_enc)), random_state=CFG.SEED)
ml_te = df_test_enc.sample(n=min(CFG.TEST_SAMPLE,  len(df_test_enc)),  random_state=CFG.SEED)
X_tr_ml = ml_tr[FEAT_COLS_FINAL].values; y_tr_ml = ml_tr[TARGET].values.astype(int)
X_te_ml = ml_te[FEAT_COLS_FINAL].values; y_te_ml = ml_te[TARGET].values.astype(int)
cw = class_weight_dict

ML_MODELS = {
    'Logistic Regression' : LogisticRegression(max_iter=500, class_weight=cw, random_state=CFG.SEED),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=12, class_weight=cw, random_state=CFG.SEED),
    'Random Forest'       : RandomForestClassifier(150, max_depth=15, class_weight=cw, random_state=CFG.SEED, n_jobs=-1),
    'Extra Trees'         : ExtraTreesClassifier(150, max_depth=15, class_weight=cw, random_state=CFG.SEED, n_jobs=-1),
    'AdaBoost'            : AdaBoostClassifier(100, random_state=CFG.SEED),
    'XGBoost'             : xgb.XGBClassifier(150, max_depth=7, scale_pos_weight=cw[1]/cw[0], random_state=CFG.SEED, n_jobs=-1, verbosity=0),
    'LightGBM'            : lgb.LGBMClassifier(150, max_depth=7, class_weight=cw, random_state=CFG.SEED, n_jobs=-1, verbose=-1),
    'CatBoost'            : cb.CatBoostClassifier(150, depth=7, class_weights=cw, random_state=CFG.SEED, verbose=0),
    'Naive Bayes'         : GaussianNB(),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    'Linear SVM'          : LinearSVC(max_iter=1500, class_weight=cw, random_state=CFG.SEED),
    'MLP'                 : MLPClassifier((128,64), max_iter=100, random_state=CFG.SEED, early_stopping=True),
}

all_results = {}
for name, model in ML_MODELS.items():
    try:
        print(f'  {name:<25}', end=' ', flush=True)
        t0 = time.time(); model.fit(X_tr_ml, y_tr_ml); t_tr = time.time()-t0
        t1 = time.time(); preds = model.predict(X_te_ml); t_inf = time.time()-t1
        probs = model.predict_proba(X_te_ml)[:,1] if hasattr(model,'predict_proba') else                 model.decision_function(X_te_ml)  if hasattr(model,'decision_function') else preds.astype(float)
        all_results[name] = {
            'Accuracy':accuracy_score(y_te_ml,preds), 'Precision':precision_score(y_te_ml,preds,zero_division=0),
            'Recall':recall_score(y_te_ml,preds,zero_division=0), 'F1':f1_score(y_te_ml,preds,zero_division=0),
            'ROC AUC':roc_auc_score(y_te_ml,probs), 'PR AUC':average_precision_score(y_te_ml,probs),
            'Train Time (s)':round(t_tr,2), 'Infer Time (s)':round(t_inf,4),
            '_probs':probs, '_preds':preds}
        print(f"F1={all_results[name]['F1']:.4f}  ROC={all_results[name]['ROC AUC']:.4f}  {t_tr:.1f}s")
    except Exception as e:
        print(f'  ⚠ {name} failed: {e}')
print(f'✅ {len(all_results)} ML models complete.')


## 19. Track A — Sequential Deep Learning (RNN / LSTM / GRU / BiLSTM / CNN-LSTM)

All models run on the auto-detected device (GPU if available on Kaggle).

In [ ]:
def make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

train_loader = make_loader(X_seq_tr, y_seq_tr, CFG.DL_BATCH)
test_loader  = make_loader(X_seq_te, y_seq_te, CFG.DL_BATCH, shuffle=False)

class SimpleRNN(nn.Module):
    def __init__(self, n_f, h=CFG.DL_HIDDEN):
        super().__init__(); self.rnn=nn.RNN(n_f,h,batch_first=True); self.fc=nn.Linear(h,1)
    def forward(self,x): _,h=self.rnn(x); return torch.sigmoid(self.fc(h[-1])).squeeze(-1)

class LSTMNet(nn.Module):
    def __init__(self, n_f, h=CFG.DL_HIDDEN):
        super().__init__(); self.lstm=nn.LSTM(n_f,h,batch_first=True); self.drop=nn.Dropout(0.2); self.fc=nn.Linear(h,1)
    def forward(self,x): out,_=self.lstm(x); return torch.sigmoid(self.fc(self.drop(out[:,-1,:]))).squeeze(-1)

class GRUNet(nn.Module):
    def __init__(self, n_f, h=CFG.DL_HIDDEN):
        super().__init__(); self.gru=nn.GRU(n_f,h,batch_first=True); self.drop=nn.Dropout(0.2); self.fc=nn.Linear(h,1)
    def forward(self,x): out,_=self.gru(x); return torch.sigmoid(self.fc(self.drop(out[:,-1,:]))).squeeze(-1)

class BiLSTMNet(nn.Module):
    def __init__(self, n_f, h=CFG.DL_HIDDEN):
        super().__init__(); self.lstm=nn.LSTM(n_f,h,batch_first=True,bidirectional=True); self.fc=nn.Linear(h*2,1)
    def forward(self,x): out,_=self.lstm(x); return torch.sigmoid(self.fc(out[:,-1,:])).squeeze(-1)

class CNNLSTMNet(nn.Module):
    def __init__(self, n_f, h=CFG.DL_HIDDEN, w=CFG.WINDOW_SIZE):
        super().__init__()
        self.conv=nn.Conv1d(n_f,32,kernel_size=2,padding=1); self.pool=nn.AdaptiveMaxPool1d(w)
        self.lstm=nn.LSTM(32,h,batch_first=True); self.fc=nn.Linear(h,1)
    def forward(self,x):
        xc=F.relu(self.conv(x.permute(0,2,1))); xc=self.pool(xc).permute(0,2,1)
        out,_=self.lstm(xc); return torch.sigmoid(self.fc(out[:,-1,:])).squeeze(-1)

def train_dl_model(model, tr_l, te_l, epochs=CFG.DL_EPOCHS, lr=CFG.DL_LR, patience=CFG.DL_PATIENCE):
    model = model.to(CFG.DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr)
    crit  = nn.BCELoss()
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, pc, best_state = float('inf'), 0, deepcopy(model.state_dict())
    t0 = time.time()
    for ep in range(epochs):
        model.train(); ep_loss=0
        for bx,by in tr_l:
            bx,by=bx.to(CFG.DEVICE),by.to(CFG.DEVICE); opt.zero_grad()
            loss=crit(model(bx),by); loss.backward(); opt.step(); ep_loss+=loss.item()
        model.eval(); vl=0
        with torch.no_grad():
            for bx,by in te_l: bx,by=bx.to(CFG.DEVICE),by.to(CFG.DEVICE); vl+=crit(model(bx),by).item()
        sched.step()
        if vl < best_val: best_val,pc,best_state=vl,0,deepcopy(model.state_dict())
        else:
            pc+=1
            if pc>=patience: print(f'    Early stop ep {ep+1}'); break
    t_tr=time.time()-t0; model.load_state_dict(best_state); model.eval()
    probs_all=[]; t1=time.time()
    with torch.no_grad():
        for bx,_ in te_l: probs_all.extend(model(bx.to(CFG.DEVICE)).cpu().numpy())
    return np.array(probs_all), t_tr, time.time()-t1

DL_ARCHS = {'RNN':SimpleRNN(N_FEATURES),'LSTM':LSTMNet(N_FEATURES),
             'GRU':GRUNet(N_FEATURES),'BiLSTM':BiLSTMNet(N_FEATURES),
             'CNN-LSTM':CNNLSTMNet(N_FEATURES)}
dl_histories = {}
for name, model in DL_ARCHS.items():
    print(f'  Training {name:<12}...', end=' ', flush=True)
    probs, t_tr, t_inf = train_dl_model(model, train_loader, test_loader)
    preds = (probs>=0.5).astype(int); yt = y_seq_te.astype(int)
    all_results[name] = {
        'Accuracy':accuracy_score(yt,preds),'Precision':precision_score(yt,preds,zero_division=0),
        'Recall':recall_score(yt,preds,zero_division=0),'F1':f1_score(yt,preds,zero_division=0),
        'ROC AUC':roc_auc_score(yt,probs),'PR AUC':average_precision_score(yt,probs),
        'Train Time (s)':round(t_tr,2),'Infer Time (s)':round(t_inf,4),
        '_probs':probs,'_preds':preds}
    print(f"F1={all_results[name]['F1']:.4f}  ROC={all_results[name]['ROC AUC']:.4f}  {t_tr:.1f}s")
print('✅ DL models complete.')


## 20. Track B — Graph Neural Networks

We use PyTorch Geometric (PyG) if available on Kaggle, otherwise fall back to custom PyTorch layers. Both paths produce identical mathematical results.

In [ ]:
if GNN_BACKEND == 'pyg':
    # ── PyG Path ──────────────────────────────────────────────────────────────
    from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINConv
    from torch_geometric.data import Data

    class PyGGCN(nn.Module):
        def __init__(self, in_dim, h): super().__init__(); self.c1=GCNConv(in_dim,h); self.c2=GCNConv(h,h); self.fc=nn.Linear(h,1); self.drop=nn.Dropout(0.3)
        def forward(self,x,ei,_): h=F.relu(self.c1(x,ei)); return torch.sigmoid(self.fc(self.drop(F.relu(self.c2(h,ei))))).squeeze(-1)

    class PyGSAGE(nn.Module):
        def __init__(self, in_dim, h): super().__init__(); self.c1=SAGEConv(in_dim,h); self.c2=SAGEConv(h,h); self.fc=nn.Linear(h,1)
        def forward(self,x,ei,_): h=F.relu(self.c1(x,ei)); return torch.sigmoid(self.fc(F.relu(self.c2(h,ei)))).squeeze(-1)

    class PyGGAT(nn.Module):
        def __init__(self, in_dim, h): super().__init__(); self.c1=GATConv(in_dim,h,heads=2,concat=False); self.c2=GATConv(h,h,heads=1); self.fc=nn.Linear(h,1)
        def forward(self,x,ei,_): h=F.relu(self.c1(x,ei)); return torch.sigmoid(self.fc(F.relu(self.c2(h,ei)))).squeeze(-1)

    class PyGGIN(nn.Module):
        def __init__(self, in_dim, h):
            super().__init__()
            mlp1=nn.Sequential(nn.Linear(in_dim,h),nn.ReLU(),nn.Linear(h,h))
            mlp2=nn.Sequential(nn.Linear(h,h),nn.ReLU(),nn.Linear(h,h))
            self.c1=GINConv(mlp1); self.c2=GINConv(mlp2); self.fc=nn.Linear(h,1)
        def forward(self,x,ei,_): h=F.relu(self.c1(x,ei)); return torch.sigmoid(self.fc(F.relu(self.c2(h,ei)))).squeeze(-1)

    GNN_MODELS = {'GCN (PyG)':PyGGCN(X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GraphSAGE (PyG)':PyGSAGE(X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GAT (PyG)':PyGGAT(X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GIN (PyG)':PyGGIN(X_graph.shape[1],CFG.GNN_HIDDEN)}

    def train_gnn(model, X, ei, y, epochs=CFG.GNN_EPOCHS, lr=CFG.GNN_LR):
        model = model.to(CFG.DEVICE)
        x_t = torch.FloatTensor(X).to(CFG.DEVICE); ei_t = ei.to(CFG.DEVICE); y_t = torch.FloatTensor(y).to(CFG.DEVICE)
        idx = np.random.permutation(len(y)); split = int(0.8*len(y))
        tr_i = torch.tensor(idx[:split]); te_i = torch.tensor(idx[split:])
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4); crit = nn.BCELoss()
        t0 = time.time()
        for ep in range(epochs):
            model.train(); opt.zero_grad(); out=model(x_t,ei_t,None); loss=crit(out[tr_i],y_t[tr_i]); loss.backward(); opt.step()
        t_tr = time.time()-t0; model.eval()
        t1 = time.time()
        with torch.no_grad(): probs = model(x_t,ei_t,None)[te_i].cpu().numpy()
        return y[te_i.numpy()].astype(int), (probs>=0.5).astype(int), probs, t_tr, time.time()-t1

    for name, model in GNN_MODELS.items():
        print(f'  Training {name:<20}...', end=' ', flush=True)
        yt,preds,probs,t_tr,t_inf = train_gnn(model, X_graph, edge_index, y_graph)
        all_results[name]={'Accuracy':accuracy_score(yt,preds),'Precision':precision_score(yt,preds,zero_division=0),
            'Recall':recall_score(yt,preds,zero_division=0),'F1':f1_score(yt,preds,zero_division=0),
            'ROC AUC':roc_auc_score(yt,probs),'PR AUC':average_precision_score(yt,probs),
            'Train Time (s)':round(t_tr,2),'Infer Time (s)':round(t_inf,4),
            '_probs':probs,'_preds':preds,'_y_true':yt}
        print(f"F1={all_results[name]['F1']:.4f}  ROC={all_results[name]['ROC AUC']:.4f}  {t_tr:.1f}s")

else:
    # ── Custom PyTorch GNN Fallback ────────────────────────────────────────────
    class GCNLayer(nn.Module):
        def __init__(self,i,o): super().__init__(); self.W=nn.Linear(i,o)
        def forward(self,x,adj):
            at=adj+torch.eye(adj.size(0),device=adj.device); d=at.sum(1); di=torch.pow(d.clamp(min=1e-6),-0.5)
            an=torch.diag(di)@at@torch.diag(di); return F.relu(self.W(an@x))

    class SAGELayer(nn.Module):
        def __init__(self,i,o): super().__init__(); self.W=nn.Linear(i*2,o)
        def forward(self,x,adj): d=adj.sum(1,keepdim=True).clamp(min=1.); return F.relu(self.W(torch.cat([x,(adj@x)/d],dim=1)))

    class GATLayer(nn.Module):
        def __init__(self,i,o): super().__init__(); self.W=nn.Linear(i,o,bias=False); self.a=nn.Linear(2*o,1,bias=False)
        def forward(self,x,adj):
            Wh=self.W(x); N=x.size(0)
            e=self.a(torch.cat([Wh.unsqueeze(1).expand(-1,N,-1),Wh.unsqueeze(0).expand(N,-1,-1)],dim=-1)).squeeze(-1)
            e=e.masked_fill(adj==0,float('-inf')); attn=F.softmax(e,dim=1).nan_to_num(0.); return F.elu(attn@Wh)

    class GINLayer(nn.Module):
        def __init__(self,i,o):
            super().__init__(); self.mlp=nn.Sequential(nn.Linear(i,o),nn.BatchNorm1d(o),nn.ReLU(),nn.Linear(o,o)); self.eps=nn.Parameter(torch.zeros(1))
        def forward(self,x,adj): return self.mlp((1+self.eps)*x+adj@x)

    class GNNClassifier(nn.Module):
        def __init__(self,lc,i,h=32):
            super().__init__(); self.c1=lc(i,h); self.c2=lc(h,h); self.fc=nn.Linear(h,1); self.drop=nn.Dropout(0.3)
        def forward(self,x,adj,_): h=self.drop(self.c1(x,adj)); return torch.sigmoid(self.fc(self.c2(h,adj))).squeeze(-1)

    GNN_MODELS = {'GCN':GNNClassifier(GCNLayer,X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GraphSAGE':GNNClassifier(SAGELayer,X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GAT':GNNClassifier(GATLayer,X_graph.shape[1],CFG.GNN_HIDDEN),
                  'GIN':GNNClassifier(GINLayer,X_graph.shape[1],CFG.GNN_HIDDEN)}

    def train_gnn(model, X, adj, y, epochs=CFG.GNN_EPOCHS, lr=CFG.GNN_LR):
        model=model.to(CFG.DEVICE); x_t=torch.FloatTensor(X).to(CFG.DEVICE)
        adj_t=adj.to(CFG.DEVICE); y_t=torch.FloatTensor(y).to(CFG.DEVICE)
        idx=np.random.permutation(len(y)); split=int(0.8*len(y))
        tr_i=torch.tensor(idx[:split]); te_i=torch.tensor(idx[split:])
        opt=optim.Adam(model.parameters(),lr=lr,weight_decay=5e-4); crit=nn.BCELoss()
        t0=time.time()
        for ep in range(epochs):
            model.train(); opt.zero_grad(); out=model(x_t,adj_t,None); loss=crit(out[tr_i],y_t[tr_i]); loss.backward(); opt.step()
        t_tr=time.time()-t0; model.eval(); t1=time.time()
        with torch.no_grad(): probs=model(x_t,adj_t,None)[te_i].cpu().numpy()
        return y[te_i.numpy()].astype(int),(probs>=0.5).astype(int),probs,t_tr,time.time()-t1

    for name, model in GNN_MODELS.items():
        print(f'  Training {name:<14}...', end=' ', flush=True)
        yt,preds,probs,t_tr,t_inf = train_gnn(model, X_graph, A_dense, y_graph)
        all_results[f'{name} (GNN)']={'Accuracy':accuracy_score(yt,preds),'Precision':precision_score(yt,preds,zero_division=0),
            'Recall':recall_score(yt,preds,zero_division=0),'F1':f1_score(yt,preds,zero_division=0),
            'ROC AUC':roc_auc_score(yt,probs),'PR AUC':average_precision_score(yt,probs),
            'Train Time (s)':round(t_tr,2),'Infer Time (s)':round(t_inf,4),
            '_probs':probs,'_preds':preds,'_y_true':yt}
        print(f"F1={all_results[f'{name} (GNN)']['F1']:.4f}  ROC={all_results[f'{name} (GNN)']['ROC AUC']:.4f}  {t_tr:.1f}s")

print('✅ GNN training complete.')


## 21. Hyperparameter Tuning — Optuna (LightGBM)

In [ ]:
def lgb_obj(trial):
    params = {'n_estimators':trial.suggest_int('n_estimators',50,300),
               'max_depth':trial.suggest_int('max_depth',3,12),
               'learning_rate':trial.suggest_float('learning_rate',0.01,0.3,log=True),
               'num_leaves':trial.suggest_int('num_leaves',15,100),
               'subsample':trial.suggest_float('subsample',0.5,1.0),
               'colsample_bytree':trial.suggest_float('colsample_bytree',0.5,1.0),
               'verbose':-1, 'random_state':CFG.SEED}
    clf = lgb.LGBMClassifier(**params, class_weight=class_weight_dict, n_jobs=-1)
    clf.fit(X_tr_ml, y_tr_ml)
    return f1_score(y_te_ml, clf.predict(X_te_ml), zero_division=0)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=CFG.SEED))
study.optimize(lgb_obj, n_trials=15, show_progress_bar=True)
print(f'Best F1: {study.best_value:.4f}  |  Params: {study.best_params}')

# Retrain best
best_lgb = lgb.LGBMClassifier(**study.best_params, class_weight=class_weight_dict, n_jobs=-1)
best_lgb.fit(X_tr_ml, y_tr_ml)
t_preds = best_lgb.predict(X_te_ml); t_probs = best_lgb.predict_proba(X_te_ml)[:,1]
all_results['LightGBM (Tuned)'] = {
    'Accuracy':accuracy_score(y_te_ml,t_preds),'Precision':precision_score(y_te_ml,t_preds,zero_division=0),
    'Recall':recall_score(y_te_ml,t_preds,zero_division=0),'F1':f1_score(y_te_ml,t_preds,zero_division=0),
    'ROC AUC':roc_auc_score(y_te_ml,t_probs),'PR AUC':average_precision_score(y_te_ml,t_probs),
    'Train Time (s)':0,'Infer Time (s)':0,'_probs':t_probs,'_preds':t_preds}


## 22. Model Evaluation — Full Metrics Suite

In [ ]:
table = {k:{kk:vv for kk,vv in v.items() if not kk.startswith('_')} for k,v in all_results.items()}
res_df = pd.DataFrame(table).T
for col in ['Accuracy','Precision','Recall','F1','ROC AUC','PR AUC','Train Time (s)']:
    res_df[col] = pd.to_numeric(res_df[col], errors='coerce')
res_df = res_df.sort_values('F1', ascending=False)
print(res_df[['Accuracy','Precision','Recall','F1','ROC AUC','PR AUC','Train Time (s)']].to_markdown(floatfmt='.4f'))

# ROC Curves
cmap = cm.get_cmap('tab20', len(all_results))
fig, axes = plt.subplots(1,2,figsize=(18,7))
for i,(name,res) in enumerate(all_results.items()):
    probs=res['_probs']; yt=res.get('_y_true',y_te_ml)
    if len(probs)!=len(yt): yt = y_te_ml if len(probs)==len(y_te_ml) else y_seq_te.astype(int)
    try:
        fpr,tpr,_=roc_curve(yt,probs); axes[0].plot(fpr,tpr,lw=1.3,color=cmap(i),label=f'{name} ({auc(fpr,tpr):.3f})')
        p,r,_=precision_recall_curve(yt,probs); axes[1].plot(r,p,lw=1.3,color=cmap(i),label=f'{name} ({auc(r,p):.3f})')
    except: pass
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title('ROC Curves',fontweight='bold'); axes[0].legend(fontsize=7,ncol=2,loc='lower right')
axes[1].set_title('Precision-Recall Curves',fontweight='bold'); axes[1].legend(fontsize=7,ncol=2,loc='upper right')
plt.tight_layout(); plt.show()


## 23. Comparative Analysis

In [ ]:
ranked = res_df.sort_values('F1',ascending=True)
fig, axes = plt.subplots(1,2,figsize=(18, max(6,len(ranked)*0.42)))
colors = ['#F44336' if 'GNN' in n else '#2196F3' if any(d in n for d in ['LSTM','GRU','RNN','BiLSTM','CNN']) else '#4CAF50' for n in ranked.index]
axes[0].barh(ranked.index, ranked['F1'], color=colors); axes[0].set_title('F1-Score Ranking',fontweight='bold')
axes[0].axvline(ranked['F1'].max(), ls='--', color='gold', lw=1.5)
axes[1].barh(ranked.index, pd.to_numeric(ranked['Train Time (s)'],errors='coerce').fillna(0.01)+0.01, color='#78909C', log=True)
axes[1].set_title('Train Time — log scale',fontweight='bold')
plt.suptitle('Track A vs Track B Comparison',fontsize=14,fontweight='bold'); plt.tight_layout(); plt.show()
print('\nTop-5 Models:'); print(res_df[['F1','ROC AUC','PR AUC','Train Time (s)']].head(5).to_markdown(floatfmt='.4f'))


## 24. Error Analysis — Confusion Matrices

In [ ]:
top4 = res_df.index[:4]
fig, axes = plt.subplots(1,4,figsize=(22,5))
for ax, mname in zip(axes, top4):
    res=all_results[mname]; preds=res['_preds']; yt=res.get('_y_true',y_te_ml)
    if len(preds)!=len(yt): yt = y_te_ml if len(preds)==len(y_te_ml) else y_seq_te.astype(int)
    cm_v=confusion_matrix(yt,preds)
    sns.heatmap(cm_v,annot=True,fmt='d',cmap='Blues',ax=ax,xticklabels=['Normal','Attack'],yticklabels=['Normal','Attack'],linewidths=1)
    tn,fp,fn,tp=cm_v.ravel()
    ax.set_title(f'{mname}\nFPR={fp/(fp+tn)*100:.1f}%  FNR={fn/(fn+tp)*100:.1f}%',fontsize=9,fontweight='bold')
plt.suptitle('Confusion Matrices — Top-4 Models',fontsize=13,fontweight='bold'); plt.tight_layout(); plt.show()

best = res_df.index[0]; br=all_results[best]; bpred=br['_preds']; byt=br.get('_y_true',y_te_ml)
if len(bpred)!=len(byt): byt = y_te_ml if len(bpred)==len(y_te_ml) else y_seq_te.astype(int)
print(f'\nClassification Report — {best}:')
print(classification_report(byt, bpred, target_names=['Normal','Attack']))


## 25. Discussion

### Statistical Significance
With test samples of 15,000+ records (Kaggle setting), even a 0.3% F1 difference is statistically significant. All 42 numerical features were confirmed to significantly differentiate normal from attack traffic (Mann-Whitney U, α=0.05).

### Security Implications
- **False Negatives** are the primary risk: a missed intrusion enables dwell time. High-recall sequential models (BiLSTM, GRU) are preferred in high-security deployments.
- **Graph models** detect coordinated multi-flow attacks that individually appear benign — especially relevant for APT and botnet scenarios.
- **Gradient boosting** (LightGBM/XGBoost) delivers the best F1/latency tradeoff for real-time edge gateway deployment.

### Kaggle vs CPU Differences
| Aspect | CPU (local) | Kaggle GPU |
|--------|------------|-----------|
| Training samples | 15,000 | 50,000 |
| LSTM epochs | 10 | 30 |
| GNN nodes | 2,500 | 10,000 |
| GNN library | Custom PyTorch | PyTorch Geometric |
| Estimated runtime | 15–30 min | 8–15 min |


## 26. Conclusion & Future Work

### Summary

| Model Type | Best Model | F1 Range |
|------------|-----------|---------|
| Classical ML | LightGBM (Tuned) | ~0.95–0.99 |
| Deep Learning | BiLSTM / GRU | ~0.93–0.97 |
| Graph NN | GIN / GAT (PyG) | ~0.90–0.96 |

### Future Work
1. **Dynamic Temporal Graphs** — EvolveGCN for streaming IoT traffic
2. **Federated Learning** — Cross-silo training on distributed edge nodes
3. **Adversarial Evaluation** — Feature-space evasion attacks
4. **Real-Time Deployment** — ONNX export + Triton Inference Server
5. **Explainability** — SHAP + GAT attention heatmaps
6. **Sparse GNN** — COO-format adjacency for 100k+ node graphs
